# Praktikum 06 – Web-Scraping, Chunking und Retrieval-Experimente
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Mehrseitige Web-Extraktion planen, Chunking-Strategien vergleichen und Retrieval-Qualitaet mit Embeddings begruendet auswerten.

**Zielprodukt nach 90 Minuten:**
1. Ein kleiner, reproduzierbarer Korpus aus mehreren Wikipedia-Artikeln.
2. Ein Vergleich von zwei Chunking-Strategien mit denselben Suchanfragen.
3. Eine kurze Schlussfolgerung, welche Strategie fuer den gewaehlten Korpus besser funktioniert.

**Arbeitsmodus:** Die Pflichtteile laufen von oben nach unten. Die Aufgaben am Ende sind die Abgabe.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "requests": ("requests", "2.33.1"),
    "bs4": ("beautifulsoup4", "4.14.3"),
    "ollama": ("ollama", "0.6.1"),
    "numpy": ("numpy", "2.4.4"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "matplotlib": ("matplotlib", "3.10.8"),
    "pandas": ("pandas", "3.0.2"),
    "tiktoken": ("tiktoken", "0.12.0"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import numpy as np
import ollama
import tiktoken


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


def model_is_available(requested_name, available_names):
    candidates = {requested_name, f"{requested_name}:latest"}
    return any(candidate in available_names for candidate in candidates)


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
for model_name in [MODEL, EMBED_MODEL]:
    subprocess.check_call(["ollama", "pull", model_name], env=env)

payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
missing_models = [
    model_name
    for model_name in [MODEL, EMBED_MODEL]
    if not model_is_available(model_name, available_models)
]
if missing_models:
    raise RuntimeError(f"Missing local Ollama models: {', '.join(missing_models)}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Models:", MODEL, ",", EMBED_MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Mehrseitiges Scraping und Korpusaufbau (25 min)
Wir bauen einen kleinen Korpus aus mehreren thematisch verwandten Wikipedia-Artikeln auf.

**Pflichtschritte:**
- Starte mit dem KI-Artikel und folge internen Wikipedia-Links bis maximal 4 Seiten.
- Extrahiere pro Seite nur inhaltliche Abschnitte mit ausreichend Text.
- Pruefe danach, aus welchen Seiten und Abschnitten dein Korpus besteht.

**Soll-Ergebnis:** eine Tabelle mit Seiten, Abschnittstiteln und Textlaengen.

In [ ]:
import re
from collections import deque
from urllib.parse import urldefrag, urljoin, urlparse

import pandas as pd

START_URL = "https://de.wikipedia.org/wiki/K%C3%BCnstliche_Intelligenz"
ARTICLE_PREFIX = "https://de.wikipedia.org/wiki/"
MAX_PAGES = 4
MAX_LINKS_PER_PAGE = 8
MIN_SECTION_CHARS = 180
USER_AGENT = {"User-Agent": "Mozilla/5.0 (Praktikum AGI NLP)"}
PRIORITY_LINK_HINTS = ["Maschinelles_Lernen", "Neuronales_Netz", "Deep_Learning"]


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_section_title(text):
    text = normalize_text(text)
    text = re.sub(r"\[\s*Bearbeiten\s*\|\s*Quelltext bearbeiten\s*\]$", "", text)
    return normalize_text(text)


def is_allowed_article(url):
    cleaned_url, _ = urldefrag(url)
    parsed = urlparse(cleaned_url)
    if parsed.scheme not in {"http", "https"}:
        return False
    article_name = cleaned_url.removeprefix(ARTICLE_PREFIX)
    return cleaned_url.startswith(ARTICLE_PREFIX) and ":" not in article_name


def link_priority(url):
    lowered_url = url.lower()
    for index, hint in enumerate(PRIORITY_LINK_HINTS):
        if hint.lower() in lowered_url:
            return index
    return len(PRIORITY_LINK_HINTS)


def extract_links(soup, base_url):
    links = []
    seen = set()
    for tag in soup.select(".mw-parser-output a[href]"):
        candidate_url = urljoin(base_url, tag["href"])
        candidate_url, _ = urldefrag(candidate_url)
        if not is_allowed_article(candidate_url):
            continue
        if candidate_url in seen:
            continue
        seen.add(candidate_url)
        links.append(candidate_url)
    links.sort(key=lambda url: (link_priority(url), url))
    return links


def extract_section_rows(section, page_title, page_url, rows):
    heading_container = next(
        (
            child
            for child in section.children
            if getattr(child, "name", None) == "div" and "mw-heading" in (child.get("class") or [])
        ),
        None,
    )

    section_title = "Einleitung"
    if heading_container is not None:
        heading = heading_container.find(re.compile(r"^h[1-6]$"), recursive=False)
        if heading is not None:
            section_title = clean_section_title(heading.get_text(" ", strip=True))

    text_parts = []
    for child in section.children:
        if not getattr(child, "name", None):
            continue
        if child == heading_container or child.name == "section":
            continue
        if child.name == "div" and "hauptartikel" in (child.get("class") or []):
            continue
        if child.name in {"p", "ul", "ol", "blockquote", "dl"}:
            block_text = normalize_text(child.get_text(" ", strip=True))
            if block_text:
                text_parts.append(block_text)

    text = " ".join(text_parts)
    if len(text) >= MIN_SECTION_CHARS:
        rows.append(
            {
                "page_title": page_title,
                "page_url": page_url,
                "section_title": section_title,
                "content": text,
                "char_count": len(text),
            }
        )

    for nested_section in section.find_all("section", recursive=False):
        extract_section_rows(nested_section, page_title, page_url, rows)


def crawl_articles(start_url, max_pages=MAX_PAGES):
    queue = deque([start_url])
    visited_urls = []
    visited_set = set()
    rows = []

    while queue and len(visited_urls) < max_pages:
        current_url = queue.popleft()
        if current_url in visited_set:
            continue

        response = requests.get(current_url, headers=USER_AGENT, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        content_root = soup.find(class_="mw-parser-output")
        if content_root is None:
            raise RuntimeError(f"Could not find article content for {current_url}")

        title_tag = soup.find("h1")
        page_title = normalize_text(title_tag.get_text(" ", strip=True)) if title_tag else current_url.rsplit("/", 1)[-1]

        top_sections = content_root.find_all("section", recursive=False)
        if not top_sections:
            raise RuntimeError(f"No top-level sections found for {current_url}")

        for top_section in top_sections:
            extract_section_rows(top_section, page_title, current_url, rows)

        visited_urls.append(current_url)
        visited_set.add(current_url)

        for link in extract_links(soup, current_url):
            if link in visited_set or link in queue:
                continue
            queue.append(link)
            if len(queue) >= MAX_LINKS_PER_PAGE:
                break

    if len(visited_urls) < max_pages:
        raise RuntimeError(f"Only crawled {len(visited_urls)} pages although {max_pages} were requested.")
    if not rows:
        raise RuntimeError("The crawl did not yield any suitable text sections.")

    return pd.DataFrame(rows), visited_urls


sections_df, crawled_urls = crawl_articles(START_URL)
sections_df = sections_df.sort_values(["page_title", "section_title"]).reset_index(drop=True)

print("Gecrawlte Seiten:")
for url in crawled_urls:
    print(f"- {url}")

print(f"\nExtrahierte Abschnitte: {len(sections_df)}")
print(sections_df[["page_title", "section_title", "char_count"]].head(12).to_string(index=False))

In [ ]:
page_summary = (
    sections_df.groupby("page_title")
    .agg(anzahl_abschnitte=("content", "count"), mittlere_zeichen=("char_count", "mean"))
    .reset_index()
    .sort_values("anzahl_abschnitte", ascending=False)
    .reset_index(drop=True)
)
page_summary["mittlere_zeichen"] = page_summary["mittlere_zeichen"].round(0).astype(int)

print(page_summary.to_string(index=False))

example_row = sections_df.iloc[0]
print("\nBeispielabschnitt:")
print(f"{example_row['page_title']} | {example_row['section_title']}")
print(example_row["content"][:600] + "...")

## Teil 2 – Chunking-Strategien vergleichen (30 min)
Wir vergleichen ein tokenbasiertes Sliding Window mit einem einfachen Zeichenfenster.

**Pflichtschritte:**
- Zerlege denselben Korpus mit zwei unterschiedlichen Strategien.
- Vergleiche Anzahl, Groesse und mittlere Tokenzahl der Chunks.
- Nutze danach identische Suchanfragen fuer beide Varianten.

**Soll-Ergebnis:** eine kleine Vergleichstabelle und eine erste Vermutung, welche Strategie fuer Retrieval besser geeignet ist.

In [ ]:
ENC = tiktoken.get_encoding("cl100k_base")


def token_chunker(text, max_tokens=140, overlap=30):
    step = max_tokens - overlap
    if step <= 0:
        raise RuntimeError("Token chunking requires overlap < max_tokens.")
    tokens = ENC.encode(text)
    return [ENC.decode(tokens[start : start + max_tokens]) for start in range(0, len(tokens), step)]


def char_chunker(text, chunk_size=420, overlap=80):
    step = chunk_size - overlap
    if step <= 0:
        raise RuntimeError("Character chunking requires overlap < chunk_size.")
    return [text[start : start + chunk_size] for start in range(0, len(text), step)]


def build_chunk_table(df, strategy_name, chunk_fn):
    rows = []
    for record in df.to_dict("records"):
        chunks = chunk_fn(record["content"])
        for chunk_index, chunk in enumerate(chunks, start=1):
            cleaned_chunk = normalize_text(chunk)
            if len(cleaned_chunk) < 80:
                continue
            rows.append(
                {
                    "strategy": strategy_name,
                    "page_title": record["page_title"],
                    "section_title": record["section_title"],
                    "chunk_index": chunk_index,
                    "text": cleaned_chunk,
                    "char_count": len(cleaned_chunk),
                    "token_count": len(ENC.encode(cleaned_chunk)),
                }
            )
    if not rows:
        raise RuntimeError(f"Chunking strategy {strategy_name} produced no usable chunks.")
    return pd.DataFrame(rows)


token_chunks = build_chunk_table(sections_df, "token_sliding", token_chunker)
char_chunks = build_chunk_table(sections_df, "char_window", char_chunker)

comparison = pd.DataFrame(
    [
        {
            "strategie": "token_sliding",
            "chunks": len(token_chunks),
            "mittlere_zeichen": int(token_chunks["char_count"].mean()),
            "mittlere_tokens": int(token_chunks["token_count"].mean()),
        },
        {
            "strategie": "char_window",
            "chunks": len(char_chunks),
            "mittlere_zeichen": int(char_chunks["char_count"].mean()),
            "mittlere_tokens": int(char_chunks["token_count"].mean()),
        },
    ]
)

print(comparison.to_string(index=False))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

RETRIEVAL_QUERIES = [
    {
        "query": "Was ist maschinelles Lernen?",
        "expected_pages": {"Maschinelles Lernen", "Künstliche Intelligenz"},
    },
    {
        "query": "Wie funktionieren neuronale Netze?",
        "expected_pages": {"Neuronales Netz", "Künstliche Intelligenz"},
    },
    {
        "query": "Welche Rolle spielt Deep Learning?",
        "expected_pages": {"Deep Learning", "Neuronales Netz"},
    },
]


def embed_texts(texts):
    return np.array([ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"] for text in texts])


def evaluate_strategy(chunks_df, queries, top_k=3):
    embeddings = embed_texts(chunks_df["text"].tolist())
    rows = []
    for item in queries:
        query_embedding = np.array(
            ollama.embeddings(model=EMBED_MODEL, prompt=item["query"])["embedding"]
        ).reshape(1, -1)
        similarities = cosine_similarity(query_embedding, embeddings)[0]
        top_indices = similarities.argsort()[::-1][:top_k]
        retrieved_pages = chunks_df.iloc[top_indices]["page_title"].tolist()
        rows.append(
            {
                "query": item["query"],
                "treffer": "JA" if any(page in item["expected_pages"] for page in retrieved_pages) else "NEIN",
                "seiten": ", ".join(retrieved_pages),
            }
        )
    return pd.DataFrame(rows)


token_eval = evaluate_strategy(token_chunks, RETRIEVAL_QUERIES)
char_eval = evaluate_strategy(char_chunks, RETRIEVAL_QUERIES)

print("Token-basiertes Chunking")
print(token_eval.to_string(index=False))

print("\nZeichenbasiertes Chunking")
print(char_eval.to_string(index=False))

## Teil 3 – Vektorraum und Retrieval-Interpretation (20 min)
Wir visualisieren eine kleine Aehnlichkeitsmatrix und gleichen sie mit den Retrieval-Ergebnissen ab.

**Leitfragen:**
- Bilden thematisch aehnliche Chunks im Vektorraum sichtbare Cluster?
- Deckt sich die Matrix mit der beobachteten Trefferquote?
- Welche Fehler entstehen durch zu grobe oder zu feine Chunks?

In [ ]:
subset = token_chunks.head(15).copy()
vectors = embed_texts(subset["text"].tolist())
similarity_matrix = cosine_similarity(vectors)

plt.figure(figsize=(7, 6))
plt.imshow(similarity_matrix, cmap="viridis")
plt.colorbar(label="Cosine Similarity")
plt.title("Aehnlichkeitsmatrix fuer 15 Token-Chunks")
plt.xlabel("Chunk-Index")
plt.ylabel("Chunk-Index")
plt.show()

summary = pd.DataFrame(
    [
        {
            "strategie": "token_sliding",
            "trefferquote_top3": round((token_eval["treffer"] == "JA").mean(), 2),
        },
        {
            "strategie": "char_window",
            "trefferquote_top3": round((char_eval["treffer"] == "JA").mean(), 2),
        },
    ]
)

print(summary.to_string(index=False))

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Notiere die Zahl der gecrawlten Seiten und der extrahierten Abschnitte.
2. Gib die Trefferquote beider Chunking-Strategien fuer die drei Suchfragen an.
3. Begruende in 3 bis 5 Saetzen, welche Strategie fuer diesen Korpus geeigneter ist.

**Transferaufgaben:**
1. Erhoehe die Crawling-Tiefe auf 5 Seiten. Veraendert sich die Trefferquote?
2. Implementiere ein Re-Ranking, das kuerzere Chunks bei gleichem Score bevorzugt.
3. Formuliere eine eigene Suchfrage, fuer die beide Strategien scheitern, und erklaere warum.